# Kinthread: score readers on real dev questions
Same reading settings for every checkpoint (extra readings voted, hybrid exact clauses). Dev only; the locked test is scored once, separately, on the final model.

In [ ]:
import os, torch
print('GPU' if torch.cuda.is_available() else 'CPU only (slower, still fine for scoring)')
REPO = '/content/Kinthread-llm-model'
if not os.path.exists(REPO):
    !git clone --depth 1 -b v9-reader https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
from google.colab import drive
drive.mount('/content/drive')
BASE = next(b for b in ('/content/drive/MyDrive/kinthread_shared', '/content/drive/MyDrive/kinthread') if os.path.isdir(b))
os.makedirs('data/real_questions', exist_ok=True)
!tar -xzf {BASE}/real_questions.tgz -C data/real_questions 2>/dev/null
!ls data/real_questions | head -30

In [ ]:
# same settings for every model: extra readings voted (--adaptive 1.0) + hybrid exact clauses; real dev only
JOBS = [('loop4', 'v10-r5-loop4', '--loops 4'), ('r6a', 'v10-r6a', ''), ('r6a-rl', 'v10-r6a-rl', '')]
for name, run, extra in JOBS:
    ck = f'{BASE}/runs/{run}/best.pt'
    if not os.path.exists(ck):
        print('missing', ck); continue
    out = f'{BASE}/runs/{run}/real_dev_a1h.json'
    print('==', name, flush=True)
    !python -m eval.run_real --ckpt {ck} {extra} --split dev --adaptive 1.0 --hybrid --out {out} 2>&1 | grep -E '"dev'